# 01 – Thu thập dữ liệu và EDA

Notebook của M1. Phần đầu (mục 1–4) làm task **M1-01** (xác minh nguồn, schema) và **M1-02** (trích xuất dữ liệu thô). Phần EDA (M1-03, M1-04) sẽ nối tiếp bên dưới sau khi có file thô.

**Tải dữ liệu phải chạy trên Google Colab**: cổng dữ liệu Chicago trả lỗi 403 khi truy cập từ máy ở Việt Nam (đã thử lại ngày 18/11/2026).

Mở trên Colab (nhánh của M1, đổi thành `main` sau khi merge):
https://colab.research.google.com/github/tanbentam/KhaiPhaDuLieu/blob/m1/extract-eda/notebooks/01_data_acquisition_eda.ipynb

Trước khi chạy trên Colab:
1. Trên Google Drive có thư mục `KhaiPhaDuLieu/data` (nằm trong My Drive, hoặc là shortcut đặt trong My Drive).
2. Nhánh ghi ở biến `BRANCH` đã được đẩy lên GitHub.

Nếu file thô đã có sẵn trong `data/raw/` thì notebook không tải lại, chỉ đọc và kiểm tra.

In [ ]:
# ===== O THIET LAP (dua tren o dung chung cua notebook 00; them BRANCH va noi Drive) =====
# Colab: clone repo, cai thu vien, noi data/ voi Google Drive.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
BRANCH = "m1/extract-eda"        # doi thanh "main" sau khi nhanh M1 duoc merge
DRIVE_DIR = "KhaiPhaDuLieu"      # = MyDrive/KhaiPhaDuLieu/data
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if IN_COLAB:
    from src.colab import link_drive_data
    print("data/ ->", link_drive_data(DRIVE_DIR))   # file ghi vao data/ se nam luon tren Drive

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

## 1. Tham số

Cửa sổ thời gian lấy từ mục `extract` của `config/experiment.yaml`. Giá trị hiện tại (17/04–14/05/2023) là **đề xuất**: bốn tuần trọn từ thứ Hai đến Chủ nhật, không có ngày lễ liên bang, không đổi giờ mùa hè, tuần đầu trùng tuần pilot của M2. Nhóm chốt trước khi tải.

In [ ]:
import json
import pandas as pd
from src.config import load_config, resolve_path
from src.data import extract as E

cfg = load_config()
START, END = cfg["extract"]["start"], cfg["extract"]["end"]
RAW_DIR, MAN_DIR = resolve_path(cfg, "raw_dir"), resolve_path(cfg, "manifests_dir")
NAME = E.raw_name(START, END)
RAW_FILE = RAW_DIR / f"{NAME}.parquet"
MAN_FILE = MAN_DIR / f"{NAME}_manifest.json"
print("Cua so  :", START, "..", END)
print("File tho:", RAW_FILE, "| da co:", RAW_FILE.exists())

## 2. Xác minh nguồn và schema (M1-01)

Đọc metadata thật của cổng dữ liệu và đối chiếu 17 cột cần lấy. Thiếu cột nào thì dừng và báo tên cột. Ô này cần gọi được cổng dữ liệu, nên chỉ chạy trên Colab.

In [ ]:
if RAW_FILE.exists():
    print("File tho da co -> doc schema da luu luc tai.")
    snap = json.loads((MAN_DIR / "schema_snapshot.json").read_text(encoding="utf-8"))
elif IN_COLAB:
    meta = E.fetch_metadata()
    E.check_columns(meta)
    snap = E.schema_snapshot(meta)
    print("Bo du lieu:", snap["name"], "|", snap["n_columns"], "cot | nguon cap nhat:", snap["rows_updated_at_utc"])
else:
    snap = None
    print("May ca nhan va chua co file tho. Chay notebook nay tren Colab, roi tai", RAW_FILE.name,
          "va thu muc manifests tu Drive ve data/.")

if snap:
    dd = E.data_dictionary(snap)
    display(dd[dd.used_by_m1][["field_name", "internal_name", "source_type", "unit", "role"]])

## 3. Đếm và tải dữ liệu thô (M1-02)

`build_raw_extract` đếm số dòng trên server, tải theo trang 50.000 dòng (sắp theo `trip_id`), rồi đối chiếu số dòng. Mỗi trang được lưu thành một file tạm, nên nếu Colab bị ngắt thì chạy lại ô này sẽ tải tiếp từ chỗ dừng. Số dòng lệch với server thì hàm báo lỗi và không ghi file.

File thô là bất biến: giữ nguyên giá trị như API trả về, không lọc dòng nào, và không bị ghi đè nếu đã tồn tại.

In [ ]:
if RAW_FILE.exists():
    man = json.loads(MAN_FILE.read_text(encoding="utf-8"))
    print("Da co file tho, khong tai lai.")
elif IN_COLAB:
    man = E.build_raw_extract(START, END, RAW_DIR, MAN_DIR)
else:
    man = None

if man:
    print("Server dem :", f"{man['rows_server_count']:,}")
    print("Tai ve     :", f"{man['rows_raw']:,}")
    print("Dung luong :", f"{man['file_size_bytes'] / 1e6:.1f} MB")
    print("sha256     :", man["sha256"])

## 4. Kiểm tra nhanh file thô

Kiểm tra file đọc được, khớp manifest và đúng cửa sổ thời gian. Đây chưa phải EDA; các con số dưới đây chỉ để xác nhận việc tải.

In [ ]:
if man:
    raw = pd.read_parquet(RAW_FILE)
    df = E.parse_types(raw)
    assert len(raw) == man["rows_raw"], "so dong khac manifest"
    assert list(raw.columns) == E.COLUMNS, "cot khac danh sach da chon"
    assert E.sha256(RAW_FILE) == man["sha256"], "sha256 khac manifest"
    ts = df["trip_start_timestamp"]
    print("So dong            :", f"{len(df):,}", "|", df.shape[1], "cot")
    print("Thoi gian          :", ts.min(), "..", ts.max(), "|", ts.dt.date.nunique(), "ngay")
    print("Phut trong du lieu :", sorted(int(m) for m in ts.dt.minute.dropna().unique()))
    print("trip_id trung      :", int(raw["trip_id"].duplicated().sum()))
    xy = df[["pickup_centroid_latitude", "pickup_centroid_longitude"]]
    print("Thieu toa do pickup:", f"{xy.isna().any(axis=1).mean():.2%}")
    print("Toa do pickup khac nhau:", f"{len(xy.dropna().drop_duplicates()):,}")
    display(raw.isna().mean().mul(100).round(2).rename("ty le thieu (%)").to_frame())
    display(df.groupby(ts.dt.date).size().rename("so chuyen").to_frame().T)

## 5. Tiếp theo

- EDA định lượng (M1-03) và EDA trực quan (M1-04) trên file thô này.
- Làm sạch, chiếu EPSG:26916, lấy mẫu phân tầng và phát hành `model_input` nằm ở notebook 02.